In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
model = LinearRegression()
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import r2_score

df = pd.read_csv("../data/hour.csv")

df.head()

In [ ]:
df["dteday"] = pd.to_datetime(df["dteday"])

df["year"] = df["dteday"].dt.year
df["month"] = df["dteday"].dt.month
df["day"] = df["dteday"].dt.day
df["dayofweek"] = df["dteday"].dt.day_of_week

df["isweekend"] = (df["dayofweek"] >= 5).astype(int)

print(df[["dteday", "year", "month", "day", "dayofweek", "isweekend"]].head())

In [ ]:
df["hr_sin"] = np.sin(2 * np.pi * df["hr"] / 24)
df["hr_cos"] = np.cos(2 * np.pi * df["hr"] / 24)

df[["hr", "hr_sin", "hr_cos"]].head(10)

In [ ]:
weather_encoded = pd.get_dummies(
    df["weathersit"], prefix="weather", drop_first=True, dtype=int
)

season_encoded = pd.get_dummies(
    df["season"], prefix="season", drop_first=True, dtype=int
)

df = pd.concat([df, weather_encoded, season_encoded], axis=1)

df.head()

In [ ]:
print("duplicate_columns :=", df.columns.duplicated().sum())
print(df.columns[df.columns.duplicated()].tolist())

### Feature Engineering Insight
Feature Engineering transforms raw data into meaningful features using date-time extraction, weekend identification, cyclic encoding, and categorical encoding to improve machine learning model learning.

In [ ]:
df.columns

instant, dteday, casual, registerd

In [ ]:
x = df[[ 'hr', 'holiday',
       'workingday', 'temp', 'atemp', 'hum', 'windspeed',
         'year', 'month', 'day', 'dayofweek',
       'isweekend', 'hr_sin', 'hr_cos', 'weather_2', 'weather_3',
       'weather_4', 'season_2', 'season_3', 'season_4']]

y = df['cnt']

print("x", x.shape)
print("y", y.shape)
print("features", x.columns.tolist())

In [ ]:
x.isnull().sum().sum(), y.isnull().sum()


Feature Selection: Selected 22 features for model training and separated cnt as the target variable. Removed ID, redundant date columns, and data leakage columns.

In [ ]:
split_index = int(len(x) * 0.8)

x_train = x.iloc[:split_index]
x_test = x.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("x train:", x_train.shape)
print("x test:", x_test.shape)
print("y train:", y_train.shape)
print("y test:", y_test.shape)

Train-Test Split: Split the dataset into 80% training data and 20% testing data using scikit-learn. Used random_state=42 for reproducibility.

In [ ]:
model.fit(x_train, y_train)

prediction = model.predict(x_test)

print("First 10 predictions:")
print(prediction[:10])

In [ ]:
feature_importance = pd.DataFrame({
    "feature" : x_train.columns,
    "coefficient" : model.coef_
})

feature_importance

In [ ]:
comparison = pd.DataFrame({
    "actual" : y_test,
    "predicted" : prediction
})

comparison.head(10)

In [ ]:
mae = mean_absolute_error(y_test, prediction)

print("MAE:", mae)

In [ ]:
mse = mean_squared_error(y_test, prediction)
print("Mean Squared Error:", mse)

In [ ]:
rmse = np.sqrt(mse)
print("Root Mean Squared Error:", rmse)

In [ ]:
r2 = r2_score(y_test, prediction)
print("R-squared:", r2)

### Model Evaluation Insight
The Linear Regression model achieved an R² score of 0.5112, explaining approximately 51.12% of the target variation. The MAE was 91.59 and RMSE was 124.41, indicating noticeable prediction errors.

In [ ]:
print(df[[
    "weather_1", "weather_2", "weather_3", "weather_4"
]].sum(axis=1).value_counts())

print(df[[
    "season_1", "season_2", "season_3", "season_4"
]].sum(axis=1).value_counts())

In [ ]:
print(df.columns.tolist())